In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Load files
# -----------------------------
output_dir = Path("data/output")
chart_dir = output_dir / "charts"
chart_dir.mkdir(parents=True, exist_ok=True)

expected = pd.read_csv(output_dir / "expected_pnl.csv")
exceptions = pd.read_csv(output_dir / "exception_report.csv")

# Parse dates
expected["date"] = pd.to_datetime(expected["date"])
exceptions["date"] = pd.to_datetime(exceptions["date"])

# -----------------------------
# 2. Daily P&L summary
# -----------------------------
daily_pnl = (
    expected.groupby("date", as_index=False)[
        ["expected_total_pnl", "realized_pnl", "unrealized_change_pnl", "fees"]
    ]
    .sum()
    .merge(
        exceptions.groupby("date", as_index=False)[["reported_pnl", "pnl_variance", "abs_pnl_variance"]].sum(),
        on="date",
        how="left"
    )
)

daily_pnl["net_pre_fee_pnl"] = daily_pnl["realized_pnl"] + daily_pnl["unrealized_change_pnl"]

daily_pnl = daily_pnl.rename(columns={
    "expected_total_pnl": "expected_total_pnl_daily",
    "reported_pnl": "reported_total_pnl_daily",
    "pnl_variance": "net_variance_daily",
    "abs_pnl_variance": "gross_abs_variance_daily"
})

# Save
daily_pnl.to_csv(output_dir / "daily_dashboard_summary.csv", index=False)

# -----------------------------
# 3. Top exception days
# -----------------------------
top_exception_days = (
    exceptions.groupby("date", as_index=False)[["abs_pnl_variance", "pnl_variance"]]
    .sum()
    .rename(columns={
        "abs_pnl_variance": "total_abs_variance",
        "pnl_variance": "net_variance"
    })
    .sort_values("total_abs_variance", ascending=False)
)

top_exception_days.to_csv(output_dir / "top_exception_days.csv", index=False)

# -----------------------------
# 4. Exception count by type
# -----------------------------
flagged = exceptions[exceptions["is_exception"] == True].copy()

exception_type_summary = (
    flagged.groupby("likely_cause", as_index=False)
    .agg(
        exception_count=("likely_cause", "count"),
        avg_abs_variance=("abs_pnl_variance", "mean"),
        max_abs_variance=("abs_pnl_variance", "max")
    )
    .sort_values("exception_count", ascending=False)
)

exception_type_summary.to_csv(output_dir / "exception_type_summary.csv", index=False)

# -----------------------------
# 5. Severity / aging summary
# -----------------------------
severity_age_summary = (
    flagged.groupby("severity", as_index=False)
    .agg(
        exception_count=("severity", "count"),
        avg_age_business_days=("age_business_days", "mean"),
        max_age_business_days=("age_business_days", "max"),
        avg_abs_variance=("abs_pnl_variance", "mean")
    )
    .sort_values("exception_count", ascending=False)
)

severity_age_summary.to_csv(output_dir / "severity_age_summary.csv", index=False)

# -----------------------------
# 6. Open exception detail
# -----------------------------
open_exception_detail = flagged[[
    "date", "instrument", "expected_pnl", "reported_pnl", "pnl_variance",
    "abs_pnl_variance", "severity", "likely_cause", "cause_confidence",
    "matched_trade_id", "implied_position_delta", "age_business_days",
    "investigation_note"
]].sort_values(["abs_pnl_variance", "age_business_days"], ascending=[False, False])

open_exception_detail.to_csv(output_dir / "open_exception_detail.csv", index=False)

# -----------------------------
# 7. Instrument-level P&L driver summary
# -----------------------------
instrument_driver_summary = (
    expected.groupby("instrument", as_index=False)[
        ["realized_pnl", "unrealized_change_pnl", "fees", "expected_total_pnl"]
    ]
    .sum()
    .rename(columns={"expected_total_pnl": "total_expected_pnl"})
)

instrument_driver_summary.to_csv(output_dir / "instrument_driver_summary.csv", index=False)

# -----------------------------
# 8. Charts
# -----------------------------

# Chart 1: Daily total P&L (expected vs reported)
plt.figure(figsize=(10, 5))
plt.plot(daily_pnl["date"], daily_pnl["expected_total_pnl_daily"], label="Expected P&L")
plt.plot(daily_pnl["date"], daily_pnl["reported_total_pnl_daily"], label="Reported P&L")
plt.axhline(0, linewidth=1)
plt.title("Daily Total P&L: Expected vs Reported")
plt.xlabel("Date")
plt.ylabel("P&L (USD)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(chart_dir / "daily_total_pnl.png", dpi=300)
plt.close()

# Chart 2: Daily net variance
plt.figure(figsize=(10, 5))
plt.bar(daily_pnl["date"], daily_pnl["net_variance_daily"])
plt.axhline(0, linewidth=1)
plt.title("Daily Net P&L Variance")
plt.xlabel("Date")
plt.ylabel("Reported - Expected (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(chart_dir / "daily_pnl_variance.png", dpi=300)
plt.close()

# Chart 3: Cumulative P&L by driver
driver_cum = daily_pnl[["date", "realized_pnl", "unrealized_change_pnl", "fees"]].copy()
driver_cum["cum_realized_pnl"] = driver_cum["realized_pnl"].cumsum()
driver_cum["cum_unrealized_change_pnl"] = driver_cum["unrealized_change_pnl"].cumsum()
driver_cum["cum_fees"] = driver_cum["fees"].cumsum()

plt.figure(figsize=(10, 5))
plt.plot(driver_cum["date"], driver_cum["cum_realized_pnl"], label="Cumulative Realized P&L")
plt.plot(driver_cum["date"], driver_cum["cum_unrealized_change_pnl"], label="Cumulative Unrealized Change")
plt.plot(driver_cum["date"], driver_cum["cum_fees"], label="Cumulative Fees")
plt.axhline(0, linewidth=1)
plt.title("Cumulative P&L by Driver")
plt.xlabel("Date")
plt.ylabel("Cumulative Value (USD)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(chart_dir / "pnl_driver_cumulative.png", dpi=300)
plt.close()

# Chart 4: Exception count by type
type_counts = flagged["likely_cause"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(type_counts.index, type_counts.values)
plt.title("Exception Count by Type")
plt.xlabel("Likely Cause")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(chart_dir / "exception_count_by_type.png", dpi=300)
plt.close()

# Chart 5: Open exceptions by severity
severity_counts = flagged["severity"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(severity_counts.index, severity_counts.values)
plt.title("Open Exceptions by Severity")
plt.xlabel("Severity")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(chart_dir / "open_exception_severity.png", dpi=300)
plt.close()

# -----------------------------
# 9. Console summary
# -----------------------------
print("Step 7 complete.")
print("\nFiles created:")
print("- daily_dashboard_summary.csv")
print("- top_exception_days.csv")
print("- exception_type_summary.csv")
print("- severity_age_summary.csv")
print("- open_exception_detail.csv")
print("- instrument_driver_summary.csv")
print("\nCharts created in data/output/charts/:")
print("- daily_total_pnl.png")
print("- daily_pnl_variance.png")
print("- pnl_driver_cumulative.png")
print("- exception_count_by_type.png")
print("- open_exception_severity.png")

print("\nTop exception days:")
print(top_exception_days.head(10))

print("\nException type summary:")
print(exception_type_summary)

print("\nSeverity / age summary:")
print(severity_age_summary)

Step 7 complete.

Files created:
- daily_dashboard_summary.csv
- top_exception_days.csv
- exception_type_summary.csv
- severity_age_summary.csv
- open_exception_detail.csv
- instrument_driver_summary.csv

Charts created in data/output/charts/:
- daily_total_pnl.png
- daily_pnl_variance.png
- pnl_driver_cumulative.png
- exception_count_by_type.png
- open_exception_severity.png

Top exception days:
         date  total_abs_variance  net_variance
40 2025-02-27             4356.75      -4356.75
35 2025-02-20             4322.50       4322.50
0  2025-01-02              425.00       -425.00
1  2025-01-03              395.25        -45.75
12 2025-01-20              362.86       -362.86
22 2025-02-03              247.85       -247.85
2  2025-01-06              235.45        235.45
19 2025-01-29              223.85        223.85
6  2025-01-10                0.00          0.00
4  2025-01-08                0.00          0.00

Exception type summary:
        likely_cause  exception_count  avg_abs_